In [24]:
import pandas as pd

file_path = "../data/raw/osnovna_sredstva.csv"

df = pd.read_csv(file_path, encoding="utf-8")
#print("Shape:",df.shape)
#print("Columns:", df.columns.tolist())
#print(df.tail(3))

#print(df["OznakaKonta"].value_counts())
#print(df["OznakaKonta"].isna().sum())

#df.head()

#print(f"Unique asset Ids: {df['IDZapisaOsnovnogSredstva'].nunique()}")
duplicate_count = df.duplicated(subset=['IDZapisaOsnovnogSredstva']).sum()
#print(f"Duplicate rows based on ID: {duplicate_count}")
df = df.drop_duplicates(subset=['IDZapisaOsnovnogSredstva'],keep='first')
#print(f"Shape after deduplication: {df.shape}")

text_cols = ['NazivSredstva','OpisMjestaTroska']
for col in text_cols:
    if col in df.columns:
        print(f"'{col}' - non-null count: {df[col].notna().sum()}")

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna('')

df['combined_text'] = df['NazivSredstva'] + " " + df['OpisMjestaTroska']
df['combined_text'] = df['combined_text'].str.replace('\s+', ' ', regex=True).str.strip()

#print(df['combined_text'].head(2))


from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['OznakaKonta'])

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

class_counts = df['label'].value_counts()
rare_classes = class_counts[class_counts < 2].index
df = df[~df['label'].isin(rare_classes)]
print(f"Removed {len(rare_classes)} classes with only 1 sample. New shape: {df.shape}")

#print(f"Number of classes: {len(label_mapping)}")
#print("First 10 mappings:",list(label_mapping.items())[:10])

X = df['combined_text']
y = df['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train)}, Validation size: {len(X_val)}")

Removed 0 rare classes. New shape: (7850, 106)
Train: 6280, Val: 1570


In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

file_path = "../data/raw/osnovna_sredstva.csv"
df = pd.read_csv(file_path, encoding="utf-8")

df = df.drop_duplicates(subset=['IDZapisaOsnovnogSredstva'],keep='first')

text_cols = ['NazivSredstva','OpisMjestaTroska']
for col in text_cols:
    df[col] = df[col].fillna('')
df['combined_text'] = (df['NazivSredstva'] + " " + df['OpisMjestaTroska']).str.replace('\s+', ' ', regex=True).str.strip()

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['OznakaKonta'])

class_counts = df['label'].value_counts()
valid_classes = class_counts[class_counts >= 2].index
df = df[df['label'].isin(valid_classes)]
print(f"Final dataset shape: {df.shape}, Unique classes: {df['label'].nunique()}")

#train - val split
X = df['combined_text']
y = df['label']

X_train, X_val, y_train, y_val = train_test_split(X,y, test_size=0.2, random_state = 42, stratify = y)
print(f"Train: {len(X_train)}, Val: {len(X_val)}")

#load tokenizer
model_name = "classla/bcms-bertic"
tokenizer = AutoTokenizer.from_pretrained(model_name)

#test sample
sample_text = X_train.iloc[0]
tokens = tokenizer(sample_text, truncation=True, padding=False, max_length= 512)

print(f"\nSample text (first 100 chars):\n{sample_text[:100]}...")
print(f"Number of tokens: {len(tokens['input_ids'])}")

Final dataset shape: (7850, 106), Unique classes: 135
Train: 6280, Val: 1570



Sample text (first 100 chars):
16-02/ U IVAN ČOLAKOVIĆ KANALIZACIJA...
Number of tokens: 18


In [3]:
class FixedAssetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = FixedAssetDataset(X_train, y_train, tokenizer, max_len=256)
val_dataset = FixedAssetDataset(X_val, y_val, tokenizer, max_len=256)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")

Train dataset size: 6280
Val dataset size: 1570


In [1]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import joblib

file_path = "../data/raw/osnovna_sredstva.csv"
df = pd.read_csv(file_path, encoding="utf-8")

df = df.drop_duplicates(subset=['IDZapisaOsnovnogSredstva'],keep='first')

text_cols = ['NazivSredstva','OpisMjestaTroska']
for col in text_cols:
    df[col] = df[col].fillna('')
df['combined_text'] = (df['NazivSredstva'] + " " + df['OpisMjestaTroska']).str.replace('\s+', ' ', regex=True).str.strip()

label_encoder_temp = LabelEncoder()
df['label_temp'] = label_encoder_temp.fit_transform(df['OznakaKonta'])

class_counts = df['label_temp'].value_counts()
valid_classes = class_counts[class_counts >= 2].index
df = df[df['label_temp'].isin(valid_classes)]

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['OznakaKonta'])

X = df['combined_text']
y = df['label']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_labels = df['label'].nunique()
print(f"Number of classes: {num_labels}")

model_name = "classla/bcms-bertic"
tokenizer = AutoTokenizer.from_pretrained(model_name)

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"]="1"
os.environ["HF_HUB_DISABLE_SYMLINKS"]="1"
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

class FixedAssetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = FixedAssetDataset(X_train, y_train, tokenizer, max_len=256)
val_dataset = FixedAssetDataset(X_val, y_val, tokenizer, max_len=256)

def compute_metrics(eval_pred):
    predictions,labels = eval_pred
    preds = np.argmax(predictions, axis = 1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels,preds,average="weighted")
    return {'accuracy':acc, 'f1':f1}

training_args = TrainingArguments(
    output_dir='../models/results',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='../models/logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

eval_results = trainer.evaluate()
print(f"Validation Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"Validation F1: {eval_results['eval_f1']:.4f}")

model.save_pretrained("../models/final_model")
tokenizer.save_pretrained("../models/final_model")
joblib.dump(label_encoder, "../models/label_encoder.pkl")
print("Model saved to ../models/final_model")

Number of classes: 135


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | Details
--------------------------------------------------+------------+--------
discriminator_predictions.dense.bias              | UNEXPECTED |        
discriminator_predictions.dense_prediction.weight | UNEXPECTED |        
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |        
discriminator_predictions.dense.weight            | UNEXPECTED |        
classifier.out_proj.weight                        | MISSING    |        
classifier.dense.bias                             | MISSING    |        
classifier.dense.weight                           | MISSING    |        
classifier.out_proj.bias                          | MISSING    |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing 

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.707500,1.469220,0.692357,0.638808
2,1.061051,1.117469,0.772611,0.732193
3,0.934964,1.021595,0.783439,0.741396


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

C:\Users\Tizian\Documents\GitHub\BERT-klasifikator\env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.934964,1.021595,3,0.783439,0.741396


Validation Accuracy: 0.7834
Validation F1: 0.7414


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ../models/final_model


###### 